# Feature Engineering

The main issue from EDA and KPI analysis is that salary is not only based on production. Rookie contracts, hardship contracts, and veteran contracts follow different salary rules.

Instead of building separate rookie, veteran, and hardship models, this notebook uses one model with contract-aware features. Separate models could show each contract group's salary structure more directly, but the current dataset is small, so splitting the data may make the models unstable.

In [38]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd

# Problem:
- The raw salary files can contain multiple salary records for the same player.
- Some records correspond to temporary or prorated contracts, such as ROS (rest-of-season), hardship, or suspended contracts.

### Solution:
 Since the raw salary files do not include contract dates, we cannot obtain the exact opening-season salary. As a practical proxy, we prioritize non-temporary salary records when available as the followings:
1. Remove temporary/prorated contract records, but if player only has temporary/prorated contract records, we will keep the highest avaliable rather than removig the player;
2. If duplicate salary records remain, we keep the highest remaining salary for each player as the full-season salary proxy;
3. validate that each player has exactly one salary record before merging.

In [39]:
# Same data pre-processing as the learnability and KPI analysis

# Load and integrate data
repo = Path.cwd().parent

# If the notebook is running from /notebooks, move one level up to the project root
if not (repo / "data").exists():
    repo = repo.parent
    
data_dir = repo / "data" / "raw"

# Load raw CSV components
adv = pd.read_csv(data_dir / "2025_advanced.csv")
per = pd.read_csv(data_dir / "2025_per_game.csv")
tot = pd.read_csv(data_dir / "2025_totals.csv")
sal = pd.read_csv(data_dir / "salary_2025.csv")
teamadv = pd.read_csv(data_dir / "2025_advanced-team.csv")
stand = pd.read_csv(data_dir / "2025_wnba_standings.csv")

# Standardize column names into a code-friendly format
def clean_col(name):
    text = str(name).strip().lower()
    text = text.replace("2025 salary", "salary")
    text = text.replace("2025 signing", "signing")
    text = text.replace("%", "pct")
    text = re.sub(r"[^0-9a-z]+", "_", text)
    return text.strip("_")

# Apply column-name standardizations and strip whitespace from text features
def clean_df(df):
    df.columns = [clean_col(col) for col in df.columns]
    for col in df.select_dtypes(include=["object", "string"]).columns:
        df[col] = df[col].astype(str).str.strip().replace({"": np.nan, "—": np.nan, "nan": np.nan})
    return df

adv, per, tot = clean_df(adv), clean_df(per), clean_df(tot)
sal, teamadv, stand = clean_df(sal), clean_df(teamadv), clean_df(stand)

# Target preparation
sal["salary"] = pd.to_numeric(sal["salary"], errors="coerce")

# Map full team names to uniform abbreviations
teammap = {
    "Atlanta Dream": "ATL", "Chicago Sky": "CHI", "Connecticut Sun": "CON",
    "Dallas Wings": "DAL", "Golden State Valkyries": "GSV", "Indiana Fever": "IND",
    "Las Vegas Aces": "LVA", "Los Angeles Sparks": "LAS", "Minnesota Lynx": "MIN",
    "New York Liberty": "NYL", "Phoenix Mercury": "PHO", "Seattle Storm": "SEA",
    "Washington Mystics": "WAS"
}

teamadv["team"] = teamadv["team"].str.replace("*", "", regex=False).str.strip().map(teammap)
stand["team_name"] = stand["team_name"].str.replace("*", "", regex=False).str.strip().map(teammap)

# Merge structural sub-tables
teamdf = teamadv.merge(stand, left_on="team", right_on="team_name", how="left", suffixes=("_adv", "_stand"))
playerdf = per.merge(adv, on=["player", "team", "pos", "g", "mp"], how="outer")
playerdf = playerdf.merge(tot, on=["player", "team", "pos", "g", "mp", "gs"], how="outer")

# Correct name spelling/encoding anomalies to maximize merge coverage
name_fix = {
    "Anastasiia Kosu": "Anastasiia Olairi Kosu", "Janelle SalaÃ¼n": "Janelle Salaun",
    "LeÃ¯la Lacan": "Leila Lacan", "Luisa GeiselsÃ¶der": "Luisa Geiselsoder",
    "Mamignan TourÃ©": "Mamignan Touré", "MariÃ¨me Badiane": "Marième Badiane",
    "Te-Hina PaoPao": "Te-Hina Paopao", "Sika KonÃ©": "Sika Kone"
}
playerdf["player"] = playerdf["player"].replace(name_fix)

# ============================================================
# Salary cleaning before constructing the final modeling table
# ============================================================
# Make sure salary is numeric
sal["salary"] = pd.to_numeric(sal["salary"], errors="coerce")

# Normalize signing status for filtering
sal["signing_clean"] = (
    sal["signing"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Check duplicated salary records before cleaning
dup_sal_before = (
    sal[sal.duplicated(subset=["player"], keep=False)]
    .sort_values(["player", "salary"])
)

print("Duplicated salary rows before cleaning:", len(dup_sal_before))
print("Players with duplicated salary records before cleaning:", dup_sal_before["player"].nunique())

display(
    dup_sal_before[["player", "salary", "signing"]].head(30)
)

# Assign lower priority to temporary or prorated contract records
temporary_contract_pattern = "ros|hardship|susp"

sal["is_temporary_salary"] = sal["signing_clean"].str.contains(
    temporary_contract_pattern,
    na=False
)

# Keep one salary record per player.
# Priority:
#   1. Prefer non-temporary salary records when available.
#   2. If a player only has temporary/prorated records, keep the highest one.
#   3. Within the selected priority group, keep the highest salary.
sal_clean = (
    sal
    .sort_values(
        ["player", "is_temporary_salary", "salary"],
        ascending=[True, True, False]
    )
    .drop_duplicates(subset=["player"], keep="first")
    .copy()
)

# Check duplicated salary records after cleaning
dup_sal_after = (
    sal_clean[sal_clean.duplicated(subset=["player"], keep=False)]
    .sort_values(["player", "salary"])
)

print("Duplicated salary rows after cleaning:", len(dup_sal_after))
print("Players with duplicated salary records after cleaning:", dup_sal_after["player"].nunique())

# Validate one salary row per player before merge
assert sal_clean["player"].duplicated().sum() == 0

# Remove signing cleaning & is_temporary_salary columns before merge
sal_clean = sal_clean.drop(columns=["signing_clean", "is_temporary_salary"])

# Preview cleaned salary table
sal_clean[["player", "salary", "signing"]].head()

# Check players with missing signing status
missing_signing = sal[sal["signing"].isna() | (sal["signing"].astype(str).str.strip() == "")]

print("Players with missing signing:", len(missing_signing))
display(missing_signing[["player", "salary", "signing"]].head(20))

# Check how many of the missing-signing players were kept in the cleaned salary table
missing_players = set(missing_signing["player"])
kept_missing_players = set(sal_clean["player"]) & missing_players

print("Missing-signing players kept in sal_clean:", len(kept_missing_players))

# Check salary players not matched to player stats before inner merge
unmatched_salary_players = sal_clean.merge(
    playerdf[["player"]],
    on="player",
    how="left",
    indicator=True
)

unmatched_salary_players = unmatched_salary_players[
    unmatched_salary_players["_merge"] == "left_only"
]

print("Salary players not matched to playerdf:", len(unmatched_salary_players))

display(
    unmatched_salary_players[["player", "salary", "signing"]].head(30)
)

# Construct final modeling table
final_df = sal_clean.merge(playerdf, on="player", how="inner", suffixes=("_sal", ""))
final_df = final_df.merge(teamdf, on="team", how="left")

Duplicated salary rows before cleaning: 73
Players with duplicated salary records before cleaning: 30


,player,salary,signing
58,Aari McDonald,6459,Hardship
57,Aari McDonald,52333,UFA
85,Aerial Powers,0,UFA
88,Aerial Powers,3975,Hardship
87,Aerial Powers,9274,Hardship
86,Aerial Powers,11924,UFA
217,Ajae Petty,1250,Hardship
216,Ajae Petty,2915,Hardship
177,Alissa Pili,10550,UFA
176,Alissa Pili,11661,UFA


Duplicated salary rows after cleaning: 0
Players with duplicated salary records after cleaning: 0
Players with missing signing: 17


,player,salary,signing
1,Napheesa Collier,214284,NaN
5,Dearica Hamby,202000,NaN
11,Jackie Young,169950,NaN
12,Caitlin Clark,78066,NaN
16,Arike Ogunbowale,249032,NaN
26,Kayla Thornton,132000,NaN
32,Ariel Atkins,223000,NaN
49,Leila Lacan,72455,NaN
68,Leonie Fiebich,68595,NaN
95,Carla Leite,72455,NaN


Missing-signing players kept in sal_clean: 17
Salary players not matched to playerdf: 2


,player,salary,signing
56,Georgia Amoore,75643,Rookie
87,Katie Lou Samuelson,95835,UFA


In [40]:
for name, df in {"per": per, "adv": adv, "tot": tot}.items():
    check = df[df["player"].isin(["Georgia Amoore", "Katie Lou Samuelson"])]
    print(name, len(check))
    display(check[["player", "team"]].head())

per 0


,player,team


adv 0


,player,team


tot 0


,player,team


**Note: The above result reports that a small number of salary records were excluded because no matching player-stat records were available in `playerdf`.**

**The two cells below are aimed to verify if `Ajae Petty` is still in the list after we assign depulicated rule.**

In [41]:
final_df[final_df["player"].eq("Ajae Petty")][
    ["player", "team", "salary", "signing"]
]

,player,team,salary,signing
5,Ajae Petty,DAL,2915,Hardship


In [42]:
print("In sal_clean:", sal_clean["player"].eq("Ajae Petty").any())
print("In final_df:", final_df["player"].eq("Ajae Petty").any())

display(
    sal_clean[sal_clean["player"].eq("Ajae Petty")][
        ["player", "salary", "signing"]
    ]
)

display(
    final_df[final_df["player"].eq("Ajae Petty")][
        ["player", "team", "salary", "signing"]
    ]
)

In sal_clean: True
In final_df: True


,player,salary,signing
216,Ajae Petty,2915,Hardship


,player,team,salary,signing
5,Ajae Petty,DAL,2915,Hardship


In [44]:
# Check whether final modeling table still has duplicated players
dup_final = (
    final_df[final_df.duplicated(subset=["player"], keep=False)]
    .sort_values("player")
)

print("Duplicated player rows in final_df:", len(dup_final))
print("Players duplicated in final_df:", dup_final["player"].nunique())

display(
    dup_final[["player", "team", "salary", "signing"]].head(30)
)

Duplicated player rows in final_df: 0
Players duplicated in final_df: 0


,player,team,salary,signing


## Feature Ideas

The new features are grouped into four simple categories.

| Group | Features | Why |
| --- | --- | --- |
| Contract | `group`, `rookie_flag`, `hardship_flag`, `vet_flag`, `unknown_flag` | Salary rules differ by contract type. |
| Role | `avail_rate`, `start_rate`, `team_min`, `starter`, `rotation` | Salary may reflect role and availability. |
| Efficiency | `pts40`, `ast40`, `reb40`, `stocks`, `stocks40`, `pps`, `ft_rate`, `three_rate`, `ast_tov`, `ws_game`, `ws40` | Makes production easier to compare across minutes. |
| Team Context | `ts_diff`, `efg_diff`, `team_power`| Compares player efficiency with team efficiency and team strength. |
| Interaction | `ws_rookie`, `ws_vet`, `ws_hardship`, `pts_rookie`, `pts_vet`, `pts_hardship` | Allows production to matter differently for rookies and veterans. |

### Contract Features

| Feature | Formula | Rationale |
| --- | --- | --- |
| group | Grouped from signing | Simplifies raw signing status into broader contract groups because rookie, hardship, and veteran contracts follow different salary rules. |
| rookie_flag | 1 if group == "rookie", else 0 | Identifies rookie-contract players, whose salaries may be lower because of rookie-scale rules. |
| hardship_flag | 1 if group == "hardship", else 0 | Identifies hardship or temporary-contract players, whose salaries may be prorated or not reflect full-season value. |
| vet_flag | 1 if group == "veteran", else 0 | Identifies veteran-market players, whose salaries are more likely to reflect market negotiation and production. |
| unknown_flag | 1 if group == "unknown", else 0 | Marks players with missing signing status so missing contract information is not ignored. |

In [45]:
# Convert raw signing status into broader contract groups
def get_group(value):
    if pd.isna(value):
        return "unknown"
    value = str(value).strip().lower()
    if "rookie" in value:
        return "rookie"
    if "hardship" in value or "susp" in value:
        return "hardship"
    if value in ["ufa", "rfa", "core"]:
        return "veteran"
    if value in ["udfa", "reserved"]:
        return "controlled"
    return "other"

# Create binary flags so models can use contract status numerically
final_df["group"] = final_df["signing"].apply(get_group)
final_df["rookie_flag"] = (final_df["group"] == "rookie").astype(int)
final_df["hardship_flag"] = (final_df["group"] == "hardship").astype(int)
final_df["vet_flag"] = (final_df["group"] == "veteran").astype(int)
final_df["unknown_flag"] = (final_df["group"] == "unknown").astype(int)

final_df[["player", "salary", "signing", "group", "rookie_flag", "hardship_flag", "vet_flag"]].head()
# print("unknown in final_df:", len(final_df[final_df["group"] == "unknown"]))

,player,salary,signing,group,rookie_flag,hardship_flag,vet_flag
0,A'ja Wilson,200000,RFA,veteran,0,0,1
1,Aaliyah Edwards,74909,Rookie,rookie,1,0,0
2,Aaliyah Nye,69267,Rookie,rookie,1,0,0
3,Aari McDonald,52333,UFA,veteran,0,0,1
4,Aerial Powers,11924,UFA,veteran,0,0,1


### Role Features

| Feature | Formula | Rationale |
| --- | --- | --- |
| `avail_rate` | `g / 44` | Measures how much of the full 2025 regular season the player appeared in. |
| `start_rate` | `gs / g` if `g > 0`, else 0 | Measures how often the player started when she appeared in games. |
| `team_min` | `mp / 44` | Measures average minutes per team game, including games the player missed. |
| `starter` | 1 if `start_rate >= 0.5`, else 0 | Identifies players who started at least half of the games they appeared in. |
| `rotation` | 1 if mp_per_g >= 15, else 0 | Identifies regular rotation players based on playing at least 15 minutes per game. |

In [46]:
season_games = 44

# Defragment DataFrame after adding new feature columns
final_df = final_df.copy()

# Create role and availability features from games, starts, and minutes
final_df["avail_rate"] = (final_df["g"] / season_games) 
final_df["start_rate"] = np.where(final_df["g"] > 0, final_df["gs"] / final_df["g"], 0) 
final_df["team_min"] = final_df["mp"] / season_games 

# Starter if she started at least half of her games
final_df["starter"] = (final_df["start_rate"] >= 0.5).astype(int)

# Rotation player if she played at least 15 minutes per game
final_df["rotation"] = (final_df["mp_per_g"] >= 15).astype(int)

role_cols = ["avail_rate", "start_rate", "team_min", "starter", "rotation"]
final_df[["player", "team", "g", "gs", "mp_per_g", *role_cols]].head()

,player,team,g,gs,mp_per_g,avail_rate,start_rate,team_min,starter,rotation
0,A'ja Wilson,LVA,40,40,31.2,0.909091,1.000000,28.340909,1,1
1,Aaliyah Edwards,TOT,36,0,14.8,0.818182,0.000000,12.136364,0,0
2,Aaliyah Nye,LVA,44,2,15.3,1.000000,0.045455,15.250000,0,1
3,Aari McDonald,IND,20,13,26.3,0.454545,0.650000,11.931818,1,1
4,Aerial Powers,TOT,10,0,16.7,0.227273,0.000000,3.795455,0,1


### Efficiency Features

| Feature | Formula | Rationale |
|----------|----------|----------|
| `pts40` | `pts_per_g * (40 / mp_per_g)` | Standardizes scoring production to a 40-minute rate so players with different playing time can be compared more fairly. |
| `ast40` | `ast_per_g * (40 / mp_per_g)` | Standardizes assist production to a 40-minute rate. |
| `reb40` | `trb_per_g * (40 / mp_per_g)` | Standardizes rebounding production to a 40-minute rate. |
| `stocks` | `stl_per_g + blk_per_g` | Combines steals and blocks into one simple defensive activity measure. |
| `stocks40` | `stocks * (40 / mp_per_g)` | Standardizes steals plus blocks to a 40-minute rate. |
| `pps` | `pts_per_g / (fga_per_g + 0.44 * fta_per_g)` | Measures scoring efficiency by comparing points to estimated shooting possessions. |
| `ft_rate` | `fta_per_g / fga_per_g` | Measures how often a player gets to the free throw line relative to field goal attempts. |
| `three_rate` | `fg3a_per_g / fga_per_g` | Measures how much of a player's shot profile comes from three-point attempts. |
| `ast_tov` | `ast_per_g / tov_per_g` | Measures playmaking efficiency by comparing assists to turnovers. |
| `ws_game` | `ws / g` | Normalizes Win Shares by games played. |
| `ws40` | `(ws / mp) * 40` | Normalizes Win Shares to a 40-minute rate. |

In [48]:
# Division helper to avoid dividing by zero
def div(a, b):
    return np.where(pd.Series(b).astype(float) != 0, a / b, np.nan)

# Multiplier for converting per-game stats to per-40-minute rates
per40 = div(40, final_df["mp_per_g"])

# Per-40 production features to compare players with different playing time
final_df["pts40"] = final_df["pts_per_g"] * per40
final_df["ast40"] = final_df["ast_per_g"] * per40
final_df["reb40"] = final_df["trb_per_g"] * per40

# Combine steals and blocks as a simple defensive activity measure
final_df["stocks"] = final_df["stl_per_g"] + final_df["blk_per_g"]
final_df["stocks40"] = final_df["stocks"] * per40

# Scoring efficiency and shot profile features
final_df["pps"] = div(final_df["pts_per_g"], final_df["fga_per_g"] + 0.44 * final_df["fta_per_g"])
final_df["ft_rate"] = div(final_df["fta_per_g"], final_df["fga_per_g"])
final_df["three_rate"] = div(final_df["fg3a_per_g"], final_df["fga_per_g"])

# Playmaking efficiency for assists compared with turnovers
final_df["ast_tov"] = div(final_df["ast_per_g"], final_df["tov_per_g"])

# Normalize win shares by games and 40 minutes
final_df["ws_game"] = div(final_df["ws"], final_df["g"])
final_df["ws40"] = div(final_df["ws"], final_df["mp"]) * 40

eff_cols = ["pts40", "ast40", "reb40", "stocks", "stocks40", "pps", "ft_rate", "three_rate", "ast_tov", "ws_game", "ws40"]
final_df[["player", "mp_per_g", "pts_per_g", "ws", *eff_cols]].head()

,player,mp_per_g,pts_per_g,ws,pts40,ast40,reb40,stocks,stocks40,pps,ft_rate,three_rate,ast_tov,ws_game,ws40
0,A'ja Wilson,31.2,23.4,9.5,30.000000,3.974359,13.076923,3.9,5.000000,1.187094,0.442424,0.090909,1.409091,0.237500,0.304731
1,Aaliyah Edwards,14.8,5.4,0.4,14.594595,1.621622,10.000000,1.0,2.702703,1.008215,0.558140,0.069767,0.545455,0.011111,0.029963
2,Aaliyah Nye,15.3,3.8,0.0,9.934641,1.307190,3.921569,0.5,1.307190,0.932287,0.102564,0.692308,0.833333,0.000000,0.000000
3,Aari McDonald,26.3,9.8,1.4,14.904943,7.148289,3.650190,1.5,2.281369,1.108096,0.337662,0.506494,2.043478,0.070000,0.106667
4,Aerial Powers,16.7,7.4,0.5,17.724551,4.790419,9.341317,0.9,2.155689,1.008724,0.292308,0.292308,1.818182,0.050000,0.119760


### Team Context Features

These features compare each player's shooting efficiency to her team's overall efficiency, so the model can see whether a player performed above or below her team context.

| Feature | Formula | Rationale |
|----------|----------|----------|
| `ts_diff` | `ts_pct_x - ts_pct_y` | Compares a player's True Shooting % with her team's True Shooting %. Positive values indicate the player was more efficient than the team average. |
| `efg_diff` | `efg_pct_x - efg_pct_y` | Compares a player's Effective Field Goal % with her team's Effective Field Goal %. Positive values indicate the player shot more efficiently than the team average. |
| `team_power` | `mean(z(win_loss_pct), z(net_rtg), z(srs))` | Creates a simple team strength index using available team-level metrics. |

In [49]:
# _x columns = player-level stats & _y columns = team-level stats
# Compare player shooting efficiency with team shooting efficiency
if "ts_pct_x" in final_df.columns and "ts_pct_y" in final_df.columns:
    final_df["ts_diff"] = final_df["ts_pct_x"] - final_df["ts_pct_y"]
else:
    final_df["ts_diff"] = np.nan

if "efg_pct_x" in final_df.columns and "efg_pct_y" in final_df.columns:
    final_df["efg_diff"] = final_df["efg_pct_x"] - final_df["efg_pct_y"]
else:
    final_df["efg_diff"] = np.nan

# Use available team-level metrics to create a simple team strength index
strength_cols = [col for col in ["win_loss_pct", "net_rtg", "srs"] if col in final_df.columns]
if strength_cols:
    temp = final_df[strength_cols].apply(pd.to_numeric, errors="coerce")
    temp = temp.fillna(temp.median())
    # Standardize columns because they use different scales
    temp = (temp - temp.mean()) / temp.std(ddof=0).replace(0, 1)
    final_df["team_power"] = temp.mean(axis=1)
else:
    final_df["team_power"] = np.nan

team_cols = ["ts_diff", "efg_diff", "team_power"]
final_df[["player", "team", *team_cols]].head()

,player,team,ts_diff,efg_diff,team_power
0,A'ja Wilson,LVA,0.044,0.018,0.653163
1,Aaliyah Edwards,TOT,NaN,NaN,0.249504
2,Aaliyah Nye,LVA,-0.079,-0.053,0.653163
3,Aari McDonald,IND,0.001,-0.011,0.402952
4,Aerial Powers,TOT,NaN,NaN,0.249504


### Interaction Features

These features are a direct response to the contract-type issue. The same production may relate to salary differently depending on contract type. For example, a rookie and a veteran can have similar `ws` or `pts_per_g`, but their salaries may follow different rules.

| Feature | Formula | Rationale |
|----------|----------|----------|
| `ws_rookie` | `ws * rookie_flag` | Captures how Win Shares relate to salary specifically for rookie players. |
| `ws_vet` | `ws * vet_flag` | Captures how Win Shares relate to salary specifically for veteran players. |
| `ws_hardship` | `ws * hardship_flag` | Captures how Win Shares relate to salary specifically for hardship players. |
| `pts_rookie` | `pts_per_g * rookie_flag` | Captures how points per game relate to salary specifically for rookie players. |
| `pts_vet` | `pts_per_g * vet_flag` | Captures how points per game relate to salary specifically for veteran players. |
| `pts_hardship` | `pts_per_g * hardship_flag` | Captures how points per game relate to salary specifically for hardship players. |

Interaction features allow one model to learn these differences without building separate models for each contract group.

For example, in a linear regression model, the relationship can be interpreted like this:

```python
salary = base
        + b1 * ws
        + b2 * rookie_flag
        + b3 * vet_flag
        + b4 * ws_rookie
        + b5 * ws_vet

In [50]:
# Win Shares interaction for each players
final_df["ws_rookie"] = final_df["ws"] * final_df["rookie_flag"]
final_df["ws_vet"] = final_df["ws"] * final_df["vet_flag"]
final_df["ws_hardship"] = final_df["ws"] * final_df["hardship_flag"]

# Points per game interaction for each players
final_df["pts_rookie"] = final_df["pts_per_g"] * final_df["rookie_flag"]
final_df["pts_vet"] = final_df["pts_per_g"] * final_df["vet_flag"]
final_df["pts_hardship"] = final_df["pts_per_g"] * final_df["hardship_flag"]

int_cols = ["ws_rookie", "ws_vet", "ws_hardship", "pts_rookie", "pts_vet", "pts_hardship"]
final_df[["player", "group", "ws", "pts_per_g", *int_cols]].head()

,player,group,ws,pts_per_g,ws_rookie,ws_vet,ws_hardship,pts_rookie,pts_vet,pts_hardship
0,A'ja Wilson,veteran,9.5,23.4,0.0,9.5,0.0,0.0,23.4,0.0
1,Aaliyah Edwards,rookie,0.4,5.4,0.4,0.0,0.0,5.4,0.0,0.0
2,Aaliyah Nye,rookie,0.0,3.8,0.0,0.0,0.0,3.8,0.0,0.0
3,Aari McDonald,veteran,1.4,9.8,0.0,1.4,0.0,0.0,9.8,0.0
4,Aerial Powers,veteran,0.5,7.4,0.0,0.5,0.0,0.0,7.4,0.0


### PCA Features

PCA is a method that compresses several correlated numeric variables into a few summary variables. In this project, many basketball production stats overlap with each other. For example, players with high minutes often also have higher points, field goal attempts, rebounds, assists, and Win Shares. So even though there are many columns, some of them repeat similar information about overall player production.

In this notebook, PCA takes production-related variables such as `mp`, `pts`, `fg`, `fga`, `ft`, `trb`, `ast`, `ws`, `pts40`, and `ast40`, then summarizes them into two new features:

| Feature | Formula | Rationale |
|----------|----------|----------|
| `pca1` | First PCA component from selected production statistics | Summarizes the largest overall pattern in player production. |
| `pca2` | Second PCA component from selected production statistics | Summarizes the second-largest production pattern that is independent of `pca1`. |

These two features are production summary scores. They are useful for model testing and for summarizing a player's overall production profile.

One limitation is that PCA features are harder to interpret than domain-driven features. For example, `pts40` clearly means points per 40 minutes, but `pca1` is a
mixture of many production stats. Because of this, I treat PCA features as supporting features rather than the main explanation for salary.

PCA cannot handle missing values directly, so the small number of missing values in the PCA input columns were filled with the median before running PCA.

The PCA variance explained output shows how much information the first two components capture. 

In [51]:
# Select production-related numeric features to summarize with PCA
pca_cols = [
    "mp", "g", "gs", "pts", "fg", "fga", "fg3", "fg3a", "ft", "fta",
    "trb", "ast", "stl", "blk", "tov", "per", "usg_pct", "ows", "dws", "ws",
    "pts40", "ast40", "reb40", "stocks40", "pps", "ast_tov"
]
# Keep only columns that actually exist in final_df
pca_cols = [col for col in pca_cols if col in final_df.columns]

# Convert to numeric and fill missing values before PCA
x = final_df[pca_cols].apply(pd.to_numeric, errors="coerce")
x = x.fillna(x.median())

# Standardize features so large-scale stats like minutes do not dominate
x = (x - x.mean()) / x.std(ddof=0).replace(0, 1)

# Run PCA using SVD and keep the first two components
u, s, vt = np.linalg.svd(x.to_numpy(), full_matrices=False)
pcs = x.to_numpy() @ vt[:2].T

# Add PCA scores as summary production features
final_df["pca1"] = pcs[:, 0]
final_df["pca2"] = pcs[:, 1]

# Check how much variation the first two PCA components explain
var = (s ** 2) / max(len(x) - 1, 1)
var_ratio = var / var.sum()
print("PCA variance explained:", var_ratio[:2])

final_df[["player", "pca1", "pca2"]].head()

PCA variance explained: [0.56318652 0.11201836]


,player,pca1,pca2
0,A'ja Wilson,-11.477298,5.753154
1,Aaliyah Edwards,1.035665,2.073663
2,Aaliyah Nye,2.033930,-1.073359
3,Aari McDonald,0.312725,-1.389695
4,Aerial Powers,2.637995,0.800856


In this result, `pca1` explains about **56.3%** of the differences across the selected production stats, and `pca2` explains about **11.2%**. Together, they summarize about **67.5%** of the production variation.

### Final Feature List

In [52]:
# Group engineered features by their main purpose
feature_groups = {
    "contract type": [
        "group", "rookie_flag", "hardship_flag", "vet_flag", "unknown_flag"
    ],
    "role / availability": [
        "avail_rate", "start_rate", "team_min", "starter", "rotation"
    ],
    "efficiency / salary prediction": [
        "pts40", "ast40", "reb40", "stocks", "stocks40", "pps",
        "ft_rate", "three_rate", "ast_tov", "ws_game", "ws40"
    ],
    "team context": [
        "ts_diff", "efg_diff", "team_power"
    ],
    "contract interaction": [
        "ws_rookie", "ws_vet", "ws_hardship",
        "pts_rookie", "pts_vet", "pts_hardship"
    ],
    "pca summary": [
        "pca1", "pca2"
    ]
}

# Convert the grouped feature dictionary into a summary table
rows = []
for category, features in feature_groups.items():
    for feature in features:
        rows.append({
            "feature": feature,
            "category": category
        })

feature_notes = pd.DataFrame(rows)
feature_notes


,feature,category
0,group,contract type
1,rookie_flag,contract type
2,hardship_flag,contract type
3,vet_flag,contract type
4,unknown_flag,contract type
5,avail_rate,role / availability
6,start_rate,role / availability
7,team_min,role / availability
8,starter,role / availability
9,rotation,role / availability
